In [77]:
import pandas as pd

In [78]:
df = pd.read_csv('2-player-UR.csv')
df.head()

,source,treatment,group,round,player,choice,score,attendance,ac_score,threshold,num_players,room,decision
0,human,score + grid,Grupo-0001,1,405186428721292,0,0,1,25,0.5,2,Grupo-0001,0
1,human,score + grid,Grupo-0001,1,728608683559397,1,1,1,25,0.5,2,Grupo-0001,1
2,human,score + grid,Grupo-0001,2,405186428721292,1,1,1,25,0.5,2,Grupo-0001,1
3,human,score + grid,Grupo-0001,2,728608683559397,0,0,1,25,0.5,2,Grupo-0001,0
4,human,score + grid,Grupo-0001,3,405186428721292,0,0,1,25,0.5,2,Grupo-0001,0


In [79]:
df['threshold'] = 0.5
df.to_csv('2-player-UR.csv', index=False)

In [80]:
file_name = '8-player-IU.csv'
df = pd.read_csv(file_name)
df.head()

,treatment,threshold,round,player,decision,score,timeup,time,room,date,num_players,group,choice
0,downwards,0.875,1.0,589037721408239,1.0,1.0,0.0,3631.0,7,12-13-2023,8,7,1.0
1,downwards,0.875,1.0,586994455133371,1.0,1.0,0.0,4885.0,7,12-13-2023,8,7,1.0
2,downwards,0.875,1.0,810776187068695,1.0,1.0,0.0,9049.0,7,12-13-2023,8,7,1.0
3,downwards,0.875,1.0,776347950518335,0.0,0.0,0.0,4979.0,7,12-13-2023,8,7,0.0
4,downwards,0.875,1.0,754052083594053,1.0,1.0,0.0,3332.0,7,12-13-2023,8,7,1.0


In [81]:
num_players = df['num_players'].unique()[0]
for threshold, grp in df.groupby('threshold'):
    name = file_name.split('.')[0]
    aux = grp.reset_index()
    aux['threshold'] = aux['threshold'] / num_players
    aux.to_csv(f'{name}-{int(threshold)}.csv')

---

In [82]:
import numpy as np
import pandas as pd

from pathlib import Path

In [83]:
import re
from collections import defaultdict

RAW_FILE_PATTERN = re.compile(r'^\d+-player-(?:IU|UR)\.csv$')

NUM_SESSIONS_BY_PLAYERS = {2: 1, 3: 2}
DEFAULT_NUM_SESSIONS = 3


def num_sessions(num_players):
    return NUM_SESSIONS_BY_PLAYERS.get(num_players, DEFAULT_NUM_SESSIONS)


def closest_bin(x):
    bins = np.array([0.25, 0.5, 0.75])
    closest_index = np.argmin(np.abs(bins - x))
    return bins[closest_index]


def add_session_columns(df):
    """Add session (threshold block) and session_round (round within block)."""
    df = df.copy().sort_values(['group', 'num_players', 'treatment', 'threshold'])

    df['session_round'] = (
        df.groupby(['group', 'num_players', 'threshold'])['round']
        .transform(lambda rounds: rounds - rounds.min())
    )

    group_counter = defaultdict(int)
    session_map = {}
    for (group, num_players, treatment, threshold), _ in df.groupby(
        ['group', 'num_players', 'treatment', 'threshold'], sort=False
    ):
        n_thresholds = num_sessions(num_players)
        if treatment == 'downwards':
            session = n_thresholds - group_counter[(group, num_players)]
            group_counter[(group, num_players)] += 1
        elif treatment == 'upwards':
            group_counter[(group, num_players)] += 1
            session = group_counter[(group, num_players)]
        else:
            session = 1
        session_map[(group, num_players, threshold)] = session

    df['session'] = df.apply(
        lambda row: session_map[(row['group'], row['num_players'], row['threshold'])],
        axis=1,
    )
    df['session_round'] = df['session_round'].astype(int)
    return df

In [84]:
df_list = []
for file_name in Path.cwd().iterdir():
    if file_name.is_file() and RAW_FILE_PATTERN.match(file_name.name):
        print(f"Reading {file_name.name}")
        df = pd.read_csv(file_name)
        df['approx_threshold'] = df['threshold'].apply(closest_bin)
        df_list.append(df)
df = pd.concat(df_list, ignore_index=True)
df = add_session_columns(df)
df.to_csv('multi-player.csv', index=True)

Reading 8-player-IU.csv
Reading 5-player-IU.csv
Reading 6-player-IU.csv
Reading 11-player-IU.csv
Reading 7-player-IU.csv
Reading 3-player-IU.csv
Reading 4-player-IU.csv
Reading 12-player-IU.csv
Reading 2-player-UR.csv
Reading 9-player-IU.csv


In [85]:
df.columns

Index(['treatment', 'threshold', 'round', 'player', 'decision', 'score',
       'timeup', 'time', 'room', 'date', 'num_players', 'group', 'choice',
       'approx_threshold', 'source', 'attendance', 'ac_score', 'session_round',
       'session'],
      dtype='str')

In [86]:
keep_columns = ['treatment', 'threshold', 'round', 'session', 'session_round', 'player', 'decision', 'score',
       'num_players', 'group', 'choice', 'approx_threshold']
df = df[keep_columns]
df['absolute_round'] = df['round']
df['round'] = df['session_round']
del df['session_round']
df.to_csv('multi-player.csv', index=True)
df.head()


,treatment,threshold,round,session,player,decision,score,num_players,group,choice,approx_threshold,absolute_round
8910,upwards,0.25,0,1,841658927890285,0.0,0.0,4,5,0.0,0.25,1.0
8911,upwards,0.25,0,1,373455036275630,1.0,-1.0,4,5,1.0,0.25,1.0
8912,upwards,0.25,0,1,477327368780915,0.0,0.0,4,5,0.0,0.25,1.0
8913,upwards,0.25,0,1,616876846241757,1.0,-1.0,4,5,1.0,0.25,1.0
8914,upwards,0.25,1,1,841658927890285,1.0,-1.0,4,5,1.0,0.25,2.0


In [87]:
df.groupby('session')['round'].value_counts()

session  round
1        0        198
         27       198
         28       198
         29       198
         26       198
                 ... 
3        24       134
         25       134
         26       134
         28       134
         0        134
Name: count, Length: 124, dtype: int64

In [88]:
df['num_players'].value_counts()

num_players
4     2880
2     2300
8     2160
5     1800
3     1620
7     1260
12    1080
6     1080
11     990
9      810
Name: count, dtype: int64